# Chequeo de disponibilidad de datos (WDI)


**Objetivo de este notebook:**
Antes de clasificar los 975 indicadores candidats (`Relevante` + `considerar` en
`df_clasificado.xlsx`) dentro de las cuatro dimensiones teóricas, se evalúa su
**disponibilidad real de datos**: cobertura temporal (años con dato) y cobertura entre
países (cuántos países soberanos reportan la serie). Esto permite descartar indicadores
inútiles por falta de datos antes de invertir tiempo en su clasificación teórica.

**Decisión metodológica clave:** en lugar de solicitar los 975 indicadores uno por uno vía
API (lento, frágil ante *rate limits*, difícil de auditar), se descarga **una sola vez**
el archivo bulk oficial del Banco Mundial (`WDI_csv.zip`), que contiene todos los
indicadores y todos los países/agregados. El filtrado y cómputo posterior se hacen
localmente en memoria con `pandas`.


# Parte 1: Metadata original del wdi

## Bloque 1 — Importación de librerías

**Objetivo:** cargar las herramientas necesarias para descargar el archivo bulk del WDI,
descomprimirlo y organizar los resultados en tablas.

Se usa `requests` (descarga HTTP estándar, ya usado en el
notebook 01) y `zipfile`/`io` de la librería estándar para evitar dependencias adicionales
innecesarias. No se usa `wbgapi` en este paso porque ese paquete está pensado para
consultas puntuales indicador por indicador, exactamente el patrón que queremos evitar por
costo computacional/de red.



In [1]:
import requests
import zipfile
import io
from pathlib import Path

import pandas as pd
import numpy as np


## Bloque 2 — Descarga del archivo bulk del WDI

**Objetivo:** descargar una única vez el archivo `WDI_csv.zip`, que contiene la base
completa de indicadores del World Development Indicators (todos los países, todos los
años, todos los indicadores).

El archivo resultante queda guardado en disco con control de que no se vuelva a descargar si
ya existe, lo que hace el notebook reproducible y evita descargas repetidas accidentales.

**Resultado esperado:** un archivo `WDI_csv.zip` en la carpeta `data/raw/`, y un mensaje
con su tamaño en MB. Si `resp.status_code` es inferior a 200mb, revisar conexión a internet o si
cambió la URL oficial de descarga.


In [2]:
RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

WDI_ZIP_URL = "https://databankfiles.worldbank.org/public/ddpext_download/WDI_CSV.zip"
zip_path = RAW_DIR / "WDI_csv.zip"

if zip_path.exists():
    print(f"Ya existe {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB). No se vuelve a descargar.")
else:
    resp = requests.get(WDI_ZIP_URL, timeout=120)
    resp.raise_for_status()
    zip_path.write_bytes(resp.content)
    print(f"Descargado {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")


Ya existe data\raw\WDI_csv.zip (283.2 MB). No se vuelve a descargar.


## Bloque 3 — Extracción de los archivos relevantes del ZIP

**Objetivo:** extraer del ZIP únicamente los dos archivos que necesitamos: los datos
(`WDICSV.csv`) y la metadata de países (`WDICountry.csv`), sin descomprimir archivos que
no vamos a usar (notas de series, metadata de indicadores ya la tenemos en
`df_clasificado.xlsx`).

**Justificación metodológica:** `WDICountry.csv` es la fuente que usamos para distinguir
país soberano de agregado regional/de ingreso: el Banco Mundial
identifica los países soberanos porque tienen un valor no vacío en la columna `Region`;
los agregados (ej. "World", "Latin America & Caribbean", "OECD members") tienen `Region`
vacío aunque tengan un código de 3 letras igual que un país.

**Resultado esperado:** dos archivos CSV extraídos en `data/raw/`. Verificar que
`WDICountry.csv` tenga alrededor de 217 filas con `Region` no vacío.


In [3]:
with zipfile.ZipFile(zip_path) as z:
    names = z.namelist()
    data_file = [n for n in names if n.upper().endswith("WDICSV.CSV")][0]
    country_file = [n for n in names if n.upper().endswith("WDICOUNTRY.CSV")][0]
    z.extract(data_file, RAW_DIR)
    z.extract(country_file, RAW_DIR)

data_path = RAW_DIR / data_file
country_path = RAW_DIR / country_file
print(data_path, country_path)


data\raw\WDICSV.csv data\raw\WDICountry.csv


## Bloque 4 — Lista de países soberanos (exclusión de agregados)

**Objetivo:** construir la lista de códigos ISO3 de países soberanos, excluyendo
agregados regionales y de nivel de ingreso.

la unidad de análisis del TFM son países (o unidades económicas para ser precisa), no agrupaciones. 


**Resultado esperado:** una lista `paises_soberanos` con ~217 códigos ISO3. 


In [4]:
country_meta = pd.read_csv(country_path)

paises_soberanos = (
    country_meta.loc[country_meta["Region"].notna(), "Country Code"]
    .unique()
    .tolist()
)

print(f"Países soberanos identificados: {len(paises_soberanos)}")
assert "ARG" in paises_soberanos, "Argentina no está en la lista — revisar filtro."


Países soberanos identificados: 217


## Bloque 5 — Lista de indicadores candidatos (975)

**Objetivo:** extraer de `df_clasificado.xlsx` los códigos WDI de los indicadores
marcados como `Relevante` o `considerar`, que son el universo de 975 indicadores a
evaluar (decisión ya acordada en la Fase 1).

**Justificación metodológica:** se excluyen los `no_relevante` porque ya fueron
descartados por criterio teórico en el paso anterior; no tiene sentido gastar cómputo en
medir su disponibilidad.

**Resultado esperado:** una lista `indicadores_candidatos` de longitud 974 (905 + 69,
según el conteo real de tu archivo; el número "975" mencionado antes era aproximado).


In [5]:
df_clasificado = pd.read_excel("df_clasificado.xlsx")

categorias_incluidas = ["Relevante", "considerar"]
indicadores_candidatos = (
    df_clasificado.loc[
        df_clasificado["Relevancia_TFM"].isin(categorias_incluidas), "Código WDI"
    ]
    .unique()
    .tolist()
)

print(f"Indicadores candidatos: {len(indicadores_candidatos)}")


Indicadores candidatos: 974


## Bloque 6 — Carga y filtrado del bulk WDI

**Objetivo:** cargar `WDICSV.csv` y quedarnos únicamente con las filas de las unidades economicas 
(países, no agregaciones) y cuyo indicador esté en la lista de 974 candidatos.

**Resultado esperado:** un DataFrame `wdi` con como máximo 974 × 217 = 211.358 filas
(en la práctica menos, porque no todos los indicadores tienen fila para todos los
países). Verificar `wdi["Indicator Code"].nunique()` y `wdi["Country Code"].nunique()`.


In [6]:
wdi_raw = pd.read_csv(data_path, low_memory=False)

wdi = wdi_raw[
    wdi_raw["Country Code"].isin(paises_soberanos)
    & wdi_raw["Indicator Code"].isin(indicadores_candidatos)
].copy()

print(f"Filas: {len(wdi)}")
print(f"Indicadores presentes: {wdi['Indicator Code'].nunique()} / {len(indicadores_candidatos)}")
print(f"Países presentes: {wdi['Country Code'].nunique()} / {len(paises_soberanos)}")


Filas: 211358
Indicadores presentes: 974 / 974
Países presentes: 217 / 217


## Bloque 7 — Reshape de formato ancho a formato largo

**Objetivo:** transformar la tabla de formato ancho (una columna por año) a formato largo
(`Country Code`, `Indicator Code`, `Year`, `Value`), que es el formato necesario para
calcular métricas de cobertura de forma vectorizada.

**Justificación metodológica:** se conserva todo el rango de años que trae el archivo
(desde 1960) para luego evaluar cómo es la cobertura de datos en el tiempo. 
Calculando la cobertura año a año una sola vez, en el Bloque 9 vas a poder generar ambos recortes sin repetir este cómputo.

**Resultado esperado:** un DataFrame `wdi_long` en formato largo. Verificar
`wdi_long["Year"].min()` y `.max()` para confirmar el rango real que trae el archivo.


In [7]:
year_cols = [c for c in wdi.columns if c.strip().isdigit()]

wdi_long = wdi.melt(
    id_vars=["Country Code", "Indicator Code"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Value",
)
wdi_long["Year"] = wdi_long["Year"].astype(int)

print(f"Rango de años disponible: {wdi_long['Year'].min()} - {wdi_long['Year'].max()}")
print(f"Filas en formato largo: {len(wdi_long)}")


Rango de años disponible: 1960 - 2025
Filas en formato largo: 13949628


## Bloque 8 — Métricas de disponibilidad

## Disponibilidad según indicador

**Objetivo:** construir disponibilidad_indicador_full, con una fila por cada uno de los 974 indicadores candidatos, indicando qué % de países soberanos tienen al menos un dato en algún año, y el primer/último año en que ese indicador tiene dato — sin recortar por ventana temporal.


Resultado esperado: un DataFrame de 974 filas. primer_anio/ultimo_anio numéricos cuando hay dato, y el string "sin_dato" cuando el indicador no tiene ningún dato para ningún país soberano.

In [8]:
# Totales base (necesarios para todos los % de este notebook en adelante)
n_paises_total = wdi_long["Country Code"].nunique()
n_indicadores_total = len(indicadores_candidatos)

print(f"Países totales: {n_paises_total}")
print(f"Indicadores totales: {n_indicadores_total}")

Países totales: 217
Indicadores totales: 974


In [9]:
paises_con_dato_full = (
    wdi_long.dropna(subset=["Value"])
    .groupby("Indicator Code")["Country Code"]
    .nunique()
    .rename("n_paises_con_dato")
)

rango_anios_full = (
    wdi_long.dropna(subset=["Value"])
    .groupby("Indicator Code")["Year"]
    .agg(primer_anio="min", ultimo_anio="max")
)

disponibilidad_indicador_full = (
    pd.DataFrame(index=indicadores_candidatos)
    .join(paises_con_dato_full)
    .join(rango_anios_full)
)

disponibilidad_indicador_full["n_paises_con_dato"] = disponibilidad_indicador_full["n_paises_con_dato"].fillna(0)
disponibilidad_indicador_full["pct_paises_con_dato"] = (
    disponibilidad_indicador_full["n_paises_con_dato"] / n_paises_total
)

# Etiqueta categórica: los años nunca se usan aritméticamente, así que "sin_dato" es seguro acá
disponibilidad_indicador_full["primer_anio"] = disponibilidad_indicador_full["primer_anio"].fillna("sin_dato")
disponibilidad_indicador_full["ultimo_anio"] = disponibilidad_indicador_full["ultimo_anio"].fillna("sin_dato")

disponibilidad_indicador_full = disponibilidad_indicador_full.sort_values("pct_paises_con_dato")
disponibilidad_indicador_full.shape

(974, 4)

In [10]:
disponibilidad_indicador_full.head()

,n_paises_con_dato,primer_anio,ultimo_anio,pct_paises_con_dato
SM.POP.RRWA.EO,1,1960,2025,0.004608
SM.POP.OPIP.EO,2,2018,2025,0.009217
SM.POP.RRWA.EA,4,1960,2025,0.018433
SH.STA.FGMS.ZS,30,1990,2023,0.138249
SM.POP.OPIP.EA,31,2018,2025,0.142857


## Disponibilidad según país
Acá quiero mostrar la disponibilidad de datos en CADA país. 
Cuántos indicadores tengo por país. Primer y último año con datos.
Qué porcentaje de indicadores hay país

In [11]:
indicadores_con_dato = (
    wdi_long.dropna(subset=["Value"])
    .groupby("Country Code")["Indicator Code"]
    .nunique()
    .rename("n_indicadores_con_dato")
)

rango_anios_pais = (
    wdi_long.dropna(subset=["Value"])
    .groupby("Country Code")["Year"]
    .agg(primer_anio="min", ultimo_anio="max")
)

disponibilidad_pais = (
    pd.DataFrame(index=paises_soberanos)
    .join(indicadores_con_dato)
    .join(rango_anios_pais)
)

disponibilidad_pais["n_indicadores_con_dato"] = disponibilidad_pais["n_indicadores_con_dato"].fillna(0)
disponibilidad_pais["pct_indicadores_con_dato"] = (
    disponibilidad_pais["n_indicadores_con_dato"] / n_indicadores_total
)

disponibilidad_pais["primer_anio"] = disponibilidad_pais["primer_anio"].fillna("sin_dato")
disponibilidad_pais["ultimo_anio"] = disponibilidad_pais["ultimo_anio"].fillna("sin_dato")

# Region es descriptiva: se une después de calcular todo, no interviene en el cálculo
region_por_pais = country_meta.set_index("Country Code")["Region"]
disponibilidad_pais["Region"] = disponibilidad_pais.index.map(region_por_pais)
disponibilidad_pais["Region"] = disponibilidad_pais["Region"].fillna("sin_dato")

disponibilidad_pais = disponibilidad_pais.sort_values("pct_indicadores_con_dato")
disponibilidad_pais.shape

(217, 5)

In [12]:
disponibilidad_pais.head()

,n_indicadores_con_dato,primer_anio,ultimo_anio,pct_indicadores_con_dato,Region
MAF,110,1960,2025,0.112936,Latin America & Caribbean
IMN,173,1960,2025,0.177618,Europe & Central Asia
CHI,184,1960,2025,0.188912,Europe & Central Asia
MNP,187,1960,2025,0.191992,East Asia & Pacific
GIB,231,1960,2025,0.237166,Europe & Central Asia


## Tabla de disponibilidad según año

In [13]:
anios_totales = sorted(wdi_long["Year"].unique())

paises_con_dato_anio = (
    wdi_long.dropna(subset=["Value"])
    .groupby("Year")["Country Code"]
    .nunique()
    .rename("n_paises_con_dato")
)

disponibilidad_anio = pd.DataFrame(index=anios_totales).join(paises_con_dato_anio)
disponibilidad_anio["n_paises_con_dato"] = disponibilidad_anio["n_paises_con_dato"].fillna(0)
disponibilidad_anio["pct_paises_con_dato"] = disponibilidad_anio["n_paises_con_dato"] / n_paises_total

# Unión con Region solo para este cálculo (no se modifica wdi_long)
wdi_long_region = wdi_long.merge(
    country_meta[["Country Code", "Region"]], on="Country Code", how="left"
)

n_paises_por_region = (
    country_meta.loc[country_meta["Region"].notna()]
    .groupby("Region")["Country Code"]
    .nunique()
)

paises_con_dato_region_anio = (
    wdi_long_region.dropna(subset=["Value"])
    .groupby(["Year", "Region"])["Country Code"]
    .nunique()
    .unstack("Region")
)

pct_region_anio = paises_con_dato_region_anio.div(n_paises_por_region, axis=1)
pct_region_anio.columns = [f"pct_{col}" for col in pct_region_anio.columns]

disponibilidad_anio = disponibilidad_anio.join(pct_region_anio).fillna(0)
# fillna(0) aquí es correcto: son porcentajes, no etiquetas, y 0% es un valor real
# (esa región no reportó nada ese año), no un "sin_dato"

disponibilidad_anio.shape

(66, 9)

In [14]:
disponibilidad_anio.head(80)

,n_paises_con_dato,pct_paises_con_dato,pct_East Asia & Pacific,pct_Europe & Central Asia,pct_Latin America & Caribbean,pct_Middle East & North Africa,pct_North America,pct_South Asia,pct_Sub-Saharan Africa
1960,217,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1961,217,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1962,217,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1963,217,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1964,217,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...
2021,217,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2022,217,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2023,217,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2024,217,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## Bloque 9: Exportamos las tres tablas de disponibilidad a un excel

In [15]:
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path_full = OUT_DIR / "disponibilidad_full.xlsx"

with pd.ExcelWriter(out_path_full) as writer:
    disponibilidad_indicador_full.to_excel(writer, sheet_name="por_indicador")
    disponibilidad_pais.to_excel(writer, sheet_name="por_pais")
    disponibilidad_anio.to_excel(writer, sheet_name="por_anio")

print(f"Exportado: {out_path_full}")

Exportado: data\processed\disponibilidad_full.xlsx


## Graficamos distribución de los porcentajes de cobertura 


In [16]:
import plotly.express as px
import plotly.graph_objects as go

# Acá se van acumulando todas las figuras (nombre, objeto fig) para el export final a HTML
figuras_reporte = []

In [17]:
df_plot = disponibilidad_indicador_full.reset_index().rename(columns={"index": "Código WDI"})

fig_cobertura_indicador = px.histogram(
    df_plot,
    x="pct_paises_con_dato",
    nbins=50,
    hover_data=["Código WDI", "n_paises_con_dato", "primer_anio", "ultimo_anio"],
    labels={"pct_paises_con_dato": "% de países soberanos con al menos un dato"},
)

fig_cobertura_indicador.update_layout(
    title="Cobertura por indicador: ¿a cuántos países alcanza cada variable?",
    xaxis_title="% de países soberanos con al menos un dato",
    yaxis_title="Cantidad de indicadores",
    bargap=0.05,
    margin=dict(b=110),  # más espacio abajo para que entre la nota
)

fig_cobertura_indicador.add_annotation(
    text=(
        "Nota: cada barra agrupa indicadores según el % de los 217 países soberanos<br>"
        "que reportan al menos un dato en algún año (1960-2025). n=974 indicadores candidatos."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.30,
    showarrow=False,
    font=dict(size=11, color="gray"),
    align="left",
)

figuras_reporte.append(("cobertura_por_indicador", fig_cobertura_indicador))
fig_cobertura_indicador.show()

Cada barra agrupa indicadores según su % de cobertura entre países. La distribución es aproximadamente bimodal: un grupo grande se concentra en coberturas altas (~0.8–1.0), hay un grupo secundario menor en coberturas medias-bajas (~0.3–0.55), y un valle relativo alrededor de 0.55–0.65 con relativamente pocos indicadores. Ese valle es un primer candidato visual para ubicar el umbral de corte, aunque conviene confirmarlo con los percentiles exactos (`disponibilidad_indicador_full["pct_paises_con_dato"].describe()` o `.quantile(...)`) antes de fijarlo definitivamente.

## Gráfico: cobertura por país
Acá buscaremos grafica pct_indicadores_con_dato por país, ordenado y coloreado por región.

Resultado esperado: un gráfico de barras horizontales, uno por país, ordenado de menor a mayor cobertura, coloreado por Region, con hover mostrando el código de país y sus años de cobertura.

In [18]:
df_pais_plot = disponibilidad_pais.reset_index().rename(columns={"index": "Country Code"})
df_pais_plot = df_pais_plot.sort_values("pct_indicadores_con_dato")

fig_cobertura_pais = px.bar(
    df_pais_plot,
    x="pct_indicadores_con_dato",
    y="Country Code",
    color="Region",
    orientation="h",
    hover_data=["n_indicadores_con_dato", "primer_anio", "ultimo_anio"],
    labels={"pct_indicadores_con_dato": "% de indicadores con al menos un dato"},
)

fig_cobertura_pais.update_layout(
    title="Cobertura por país: ¿qué proporción de indicadores reporta cada país?",
    xaxis_title="% de indicadores con al menos un dato",
    yaxis_title="País (código ISO3)",
    height=3200,
    margin=dict(b=110),
)

fig_cobertura_pais.add_annotation(
    text=(
        "Nota: cada barra representa uno de los 217 países soberanos. El % indica cuántos<br>"
        "de los 974 indicadores candidatos tienen al menos un dato para ese país (1960-2025)."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.03,
    showarrow=False,
    font=dict(size=13, color="gray"),
    align="left",
)

figuras_reporte.append(("cobertura_por_pais", fig_cobertura_pais))
fig_cobertura_pais.show()

Los países con cobertura más baja no están distribuidos al azar: se concentran en microestados y territorios dependientes, no en países soberanos "grandes" con problemas de reporte. Aparecen sistemáticamente en la cola baja: Gibraltar (GIB), Isla de Man (IMN), Groenlandia (GRL), Islas Caimán (CYM), Islas Vírgenes (VIR), Turcas y Caicos (TCA), Aruba (ABW), Antigua y Barbuda (ATG), Granada (GRD) — todos con cobertura por debajo de ~0.4-0.5. Esto sugiere que el criterio de exclusión más defendible teóricamente no sería "región" sino algo como tamaño poblacional o estatus de territorio dependiente vs. Estado soberano pleno.
Creo que acá voy a tener que redefinir las unidades de estudio y dejar países como tales

## Gráfico: Cobertura por región
el gráfico anterior muestra cada país individualmente; este resume la dispersión dentro de cada región, que es lo que hace falta para detectar si el problema de cobertura es sistemático de una región

In [19]:
fig_dispersion_region = px.box(
    df_pais_plot,
    x="Region",
    y="pct_indicadores_con_dato",
    points="all",
    hover_data=["Country Code", "n_indicadores_con_dato"],
    labels={"pct_indicadores_con_dato": "% de indicadores con al menos un dato"},
)

fig_dispersion_region.update_layout(
    title="Dispersión de cobertura por región: ¿hay regiones sistemáticamente peor cubiertas?",
    xaxis_title="Región (Banco Mundial)",
    yaxis_title="% de indicadores con al menos un dato",
    margin=dict(b=210),
)

fig_dispersion_region.add_annotation(
    text=(
        "Nota: cada punto es un país soberano; la caja muestra la mediana y el rango<br>"
        "intercuartílico de cobertura de indicadores dentro de cada región del Banco Mundial."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.80,
    showarrow=False,
    font=dict(size=10, color="gray"),
    align="left",
)

figuras_reporte.append(("dispersion_cobertura_region", fig_dispersion_region))
fig_dispersion_region.show()

Los boxplots presentan por región, la misma información que pudimos ver en el gráfico de cobertura por país. De hecho fue realizado utilizando el mismo dataframe, pero utilizando por referencia a las regiones. Este gráfico permite apreciar nuevamente que la región geográfica en sí no parece ser el criterio que mejor discrimina cobertura baja. 

In [20]:
df_pais_plot.columns

Index(['Country Code', 'n_indicadores_con_dato', 'primer_anio', 'ultimo_anio',
       'pct_indicadores_con_dato', 'Region'],
      dtype='str')

In [21]:
df_pais_plot.Region.unique()

<ArrowStringArray>
[ 'Latin America & Caribbean',      'Europe & Central Asia',
        'East Asia & Pacific',              'North America',
         'Sub-Saharan Africa', 'Middle East & North Africa',
                 'South Asia']
Length: 7, dtype: str

## Gráfico: curva de cobertura agregada por año

In [22]:
n_anios_totales = wdi_long["Year"].nunique()
celdas_posibles_total = n_paises_total * n_indicadores_total

celdas_con_dato_anio = wdi_long.dropna(subset=["Value"]).groupby("Year").size()

df_anio_plot = disponibilidad_anio.reset_index().rename(columns={"index": "Year"})
df_anio_plot["densidad_datos"] = (
    df_anio_plot["Year"].map(celdas_con_dato_anio).fillna(0) / celdas_posibles_total
)

fig_densidad_anio = px.line(
    df_anio_plot,
    x="Year",
    y="densidad_datos",
    markers=True,
    labels={"densidad_datos": "% de celdas país-indicador con dato"},
)

fig_densidad_anio.update_layout(
    title="Sensibilidad de la cobertura: ¿qué proporción de la matriz país-indicador está completa?",
    xaxis_title="Año",
    yaxis_title="% de celdas país-indicador con dato",
    yaxis_range=[0, 1.05],
    margin=dict(b=130),
)

fig_densidad_anio.add_annotation(
    text=(
        "Nota: Este gráfico busca medir para cada año qué % de las 217×974 celdas<br>"
        "posibles país-indicador tienen realmente un valor. Es una medida de densidad, no de presencia binaria."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.42,
    showarrow=False,
    font=dict(size=12, color="gray"),
    align="left",
)

figuras_reporte.append(("densidad_panel_por_anio", fig_densidad_anio))
fig_densidad_anio.show()

**Cómo leer este gráfico:** buscá el punto en que la curva deja de subir de forma pronunciada y empieza a estabilizarse (el "codo" de la curva) — ese año es un candidato empírico razonable para el inicio del panel, porque antes de él gran parte de la matriz país-indicador está vacía y cualquier análisis dependería en exceso de imputación. Prestá atención también al extremo derecho: si la densidad cae en los últimos años (2023-2025), es esperable por rezagos de publicación del Banco Mundial, y puede llevar a recortar también el extremo superior del período, no solo el inferior. Estos valores concretos (año de quiebre, nivel de densidad en ese año) hay que leerlos directamente del gráfico al ejecutarlo — no deben asumirse de antemano.

## Gráfico: disponibilidad regional en el tiempo (heatmap)
heatmap Región × Año con el % de cobertura de cada región en cada año.

In [23]:
wdi_long_region = wdi_long.merge(
    country_meta[["Country Code", "Region"]], on="Country Code", how="left"
)

n_paises_por_region = (
    country_meta.loc[country_meta["Region"].notna()]
    .groupby("Region")["Country Code"]
    .nunique()
)
celdas_posibles_region = n_paises_por_region * n_indicadores_total

celdas_con_dato_region_anio = (
    wdi_long_region.dropna(subset=["Value"])
    .groupby(["Year", "Region"])
    .size()
    .unstack("Region")
)

densidad_region_anio = celdas_con_dato_region_anio.div(celdas_posibles_region, axis=1).fillna(0)
matriz_region_anio = densidad_region_anio.T

fig_heatmap_region_anio = px.imshow(
    matriz_region_anio,
    labels=dict(x="Año", y="Región", color="% celdas con dato"),
    x=matriz_region_anio.columns,
    y=matriz_region_anio.index,
    color_continuous_scale="RdYlGn",
    zmin=0,
    zmax=float(matriz_region_anio.values.max()),
    aspect="auto",
)

fig_heatmap_region_anio.update_layout(
    title="Densidad regional de datos en el tiempo: ¿dónde y cuándo se concentran los huecos?",
    margin=dict(b=130),
)

fig_heatmap_region_anio.add_annotation(
    text=(
        "Nota: cada celda muestra, para esa región y ese año, qué % de las celdas país-<br>"
        "indicador posibles (países de la región × 974 indicadores) tienen realmente un dato.<br>"
        "A diferencia de la versión anterior (todo verde), esta mide densidad real, no presencia binaria."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.45,
    showarrow=False,
    font=dict(size=13, color="gray"),
    align="left",
)

figuras_reporte.append(("densidad_regional_heatmap", fig_heatmap_region_anio))
fig_heatmap_region_anio.show()

**Cómo leer este heatmap:** con la métrica de densidad real (en vez de "al menos un dato"), debería verse variación de color entre regiones y a lo largo del tiempo. Prestá atención a: (i) si alguna región muestra colores sistemáticamente más rojos (menor densidad) durante todo el período — señal de que esa región necesitará más imputación o debería tratarse aparte; (ii) si la densidad de todas las regiones sube de forma pareja en algún año puntual — reforzaría la elección de ese año como inicio del panel; (iii) caídas puntuales de densidad en alguna región/año, que pueden señalar problemas de reporte específicos a investigar antes de decidir exclusiones.

## Gráfico: dispersión de cobertura por década
boxplot de la densidad real de datos (% de celdas país-indicador con dato) agrupando los años en décadas.

resume la variabilidad año a año dentro de cada década, útil para ver si hay décadas con mayor inestabilidad de cobertura (no solo el promedio, sino la dispersión).

Resultado esperado: una caja por década, de 1960s a 2020s.

In [24]:
df_anio_plot["decada"] = (df_anio_plot["Year"] // 10 * 10).astype(str) + "s"
orden_decadas = sorted(df_anio_plot["decada"].unique())

fig_dispersion_decada = px.box(
    df_anio_plot,
    x="decada",
    y="densidad_datos",
    points="all",
    category_orders={"decada": orden_decadas},
    hover_data=["Year"],
    labels={"densidad_datos": "% de celdas país-indicador con dato"},
)

fig_dispersion_decada.update_layout(
    title="Estabilidad de la densidad de datos por década",
    xaxis_title="Década",
    yaxis_title="% de celdas país-indicador con dato",
    margin=dict(b=110),
)

fig_dispersion_decada.add_annotation(
    text=(
        "Nota: cada punto es un año dentro de la década; la caja muestra la dispersión<br>"
        "de la densidad real de datos (celdas país-indicador completas) entre los años de esa década."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.30,
    showarrow=False,
    font=dict(size=10, color="gray"),
    align="left",
)

figuras_reporte.append(("dispersion_densidad_decada", fig_dispersion_decada))
fig_dispersion_decada.show()

**Cómo leer este gráfico:** compará el ancho de las cajas entre décadas. Décadas con cajas anchas indican que la densidad de datos varió mucho año a año dentro de ese período (menos confiable para tomarla completa como bloque homogéneo); décadas con cajas angostas y bien ubicadas en niveles altos son más seguras para incluir sin más chequeos. Si el nivel medio empieza a estabilizarse a partir de determinada década, es otra pieza de evidencia (junto con el "codo" del gráfico anterior) para fijar el año de inicio del panel.

## Gráfico: densidad real de datos, por indicador y por país

A diferencia del criterio binario usado antes ("¿tiene o no tiene al menos un dato?", que da 100% tanto para indicadores como para países, porque todos tienen al menos un valor en algún año), acá se mide la **densidad real**: qué proporción de las celdas país-año (para cada indicador) o indicador-año (para cada país) están efectivamente completas.

Objetivo: ver la distribución de densidad real —no solo la presencia binaria— para dimensionar cuán completo está realmente el panel más allá de que "algo" de dato exista.

Resultado esperado: dos distribuciones (por indicador y por país) del % de celdas con dato, más la densidad global del panel completo.

In [25]:
n_anios_totales = wdi_long["Year"].nunique()

celdas_indicador = wdi_long.dropna(subset=["Value"]).groupby("Indicator Code").size()
densidad_indicador = (
    celdas_indicador.reindex(indicadores_candidatos).fillna(0) / (n_paises_total * n_anios_totales)
)

celdas_pais = wdi_long.dropna(subset=["Value"]).groupby("Country Code").size()
densidad_pais = (
    celdas_pais.reindex(paises_soberanos).fillna(0) / (n_indicadores_total * n_anios_totales)
)

densidad_total = wdi_long["Value"].notna().sum() / len(wdi_long)

df_densidad = pd.concat([
    pd.DataFrame({"Código": densidad_indicador.index, "Densidad": densidad_indicador.values, "Nivel": "Por indicador"}),
    pd.DataFrame({"Código": densidad_pais.index, "Densidad": densidad_pais.values, "Nivel": "Por país"}),
])

fig_densidad_real = px.histogram(
    df_densidad,
    x="Densidad",
    facet_col="Nivel",
    nbins=30,
    labels={"Densidad": "% de celdas con dato"},
)

fig_densidad_real.update_layout(
    title=f"Densidad real del panel: proporción de celdas con dato (densidad global = {densidad_total:.1%})",
    margin=dict(b=130),
)

fig_densidad_real.add_annotation(
    text=(
        "Nota: 'Por indicador' = para cada indicador, % de las celdas país-año (217×"
        f"{n_anios_totales}) con dato. 'Por país' = para cada país, % de las celdas indicador-<br>"
        "año (974×" + str(n_anios_totales) + ") con dato. No es lo mismo que 'tener al menos un dato': mide cuán<br>"
        "completa está realmente la serie, no solo su existencia."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.45,
    showarrow=False,
    font=dict(size=10, color="gray"),
    align="left",
)

figuras_reporte.append(("densidad_real_indicador_pais", fig_densidad_real))
fig_densidad_real.show()

**Cómo leer este gráfico:** la densidad global resume qué proporción de toda la matriz país-indicador-año está efectivamente completa. Las dos distribuciones muestran si esa densidad es pareja (histogramas concentrados) o si hay indicadores/países con densidad muy baja aunque figuren como "con dato" en los gráficos anteriores. Una densidad baja y muy dispersa por indicador sugiere priorizar, para el análisis, los indicadores del extremo derecho de esa distribución; lo mismo aplica para países.

# Parte 2: Incorporando codificación M49 para regiones

## Unimos la codificación de países miembros de onu con M49 a nuestro df del BM (WDI)

In [26]:
df_regiones_onu = pd.read_excel("df_regiones_miembros_onu.xlsx")
df_regiones_onu.head()

,Member State,M49_country,ISO-alpha3,Other Names,Country or Area,M49_region,Region Name,M49_subregion,Sub-region Name,ISO-alpha2 Code
0,United States,840,USA,"USA, U.S.A., United States of America",United States of America,19,Americas,21,Northern America,US
1,Australia,36,AUS,Commonwealth of Australia,Australia,9,Oceania,53,Australia and New Zealand,AU
2,Djibouti,262,DJI,Republic of Djibouti,Djibouti,2,Africa,202,Sub-Saharan Africa,DJ
3,Ghana,288,GHA,Republic of Ghana,Ghana,2,Africa,202,Sub-Saharan Africa,GH
4,Kiribati,296,KIR,Republic of Kiribati,Kiribati,9,Oceania,57,Micronesia,KI


In [27]:
iso_onu = set(df_regiones_onu["ISO-alpha3"])

# Nueva lista: soberanos WDI que SÍ son Estados miembros de la ONU (no sobreescribe paises_soberanos)
paises_onu = [c for c in paises_soberanos if c in iso_onu]

# Auditoría 1: soberanos WDI que NO matchean con la ONU (territorios, no miembros, etc.)
wdi_sin_match_onu = sorted(set(paises_soberanos) - iso_onu)

# Auditoría 2: Estados miembros ONU que NO aparecen como soberanos en WDI (ausencia total de datos)
onu_sin_match_wdi = df_regiones_onu.loc[~df_regiones_onu["ISO-alpha3"].isin(paises_soberanos)]

print(f"Países soberanos WDI (original): {len(paises_soberanos)}")
print(f"Países ONU tras el match: {len(paises_onu)}")
print(f"\nWDI sin match en ONU ({len(wdi_sin_match_onu)}):")
print(wdi_sin_match_onu)

print(f"\nEstados ONU sin match en WDI ({len(onu_sin_match_wdi)}):")
onu_sin_match_wdi[["Member State", "ISO-alpha3"]]

Países soberanos WDI (original): 217
Países ONU tras el match: 193

WDI sin match en ONU (24):
['ABW', 'ASM', 'BMU', 'CHI', 'CUW', 'CYM', 'FRO', 'GIB', 'GRL', 'GUM', 'HKG', 'IMN', 'MAC', 'MAF', 'MNP', 'NCL', 'PRI', 'PSE', 'PYF', 'SXM', 'TCA', 'VGB', 'VIR', 'XKX']

Estados ONU sin match en WDI (0):


,Member State,ISO-alpha3


## Recalculamos

In [28]:
n_paises_total_onu = len(paises_onu)
print(f"Países totales (ONU): {n_paises_total_onu}  |  Países totales (original): {n_paises_total}")

Países totales (ONU): 193  |  Países totales (original): 217


In [29]:
wdi_long_onu = wdi_long[wdi_long["Country Code"].isin(paises_onu)]

paises_con_dato_full_onu = (
    wdi_long_onu.dropna(subset=["Value"])
    .groupby("Indicator Code")["Country Code"]
    .nunique()
    .rename("n_paises_con_dato_onu")
)

disponibilidad_indicador_full_onu = (
    pd.DataFrame(index=indicadores_candidatos)
    .join(paises_con_dato_full_onu)
)
disponibilidad_indicador_full_onu["n_paises_con_dato_onu"] = disponibilidad_indicador_full_onu["n_paises_con_dato_onu"].fillna(0)
disponibilidad_indicador_full_onu["pct_paises_con_dato_onu"] = (
    disponibilidad_indicador_full_onu["n_paises_con_dato_onu"] / n_paises_total_onu
)

In [30]:
disponibilidad_pais_onu = disponibilidad_pais.loc[paises_onu].copy()

# Anexamos columnas M49 (descriptivas, igual que se hizo con "Region" en el bloque 21)
m49_por_pais = df_regiones_onu.set_index("ISO-alpha3")[
    ["M49_region", "Region Name", "M49_subregion", "Sub-region Name"]
]
disponibilidad_pais_onu = disponibilidad_pais_onu.join(m49_por_pais)

disponibilidad_pais_onu.shape

(193, 9)

## Cobertura por país con universo ONU

In [31]:
disponibilidad_pais_onu.head()

,n_indicadores_con_dato,primer_anio,ultimo_anio,pct_indicadores_con_dato,Region,M49_region,Region Name,M49_subregion,Sub-region Name
AFG,874,1960,2025,0.897331,Middle East & North Africa,142,Asia,34,Southern Asia
AGO,938,1960,2025,0.963039,Sub-Saharan Africa,2,Africa,202,Sub-Saharan Africa
ALB,941,1960,2025,0.966119,Europe & Central Asia,150,Europe,39,Southern Europe
AND,429,1960,2025,0.440452,Europe & Central Asia,150,Europe,39,Southern Europe
ARE,806,1960,2025,0.827515,Middle East & North Africa,142,Asia,145,Western Asia


In [32]:
df_pais_plot_onu = disponibilidad_pais_onu.reset_index().rename(columns={"index": "Country Code"})
df_pais_plot_onu = df_pais_plot_onu.sort_values("pct_indicadores_con_dato")

fig_cobertura_pais_onu = px.bar(
    df_pais_plot_onu,
    x="pct_indicadores_con_dato",
    y="Country Code",
    color="Region Name",
    orientation="h",
    hover_data=["n_indicadores_con_dato", "primer_anio", "ultimo_anio", "Sub-region Name"],
    labels={"pct_indicadores_con_dato": "% de indicadores con al menos un dato"},
)

fig_cobertura_pais_onu.update_layout(
    title="Cobertura por país (universo ONU, M49): ¿qué proporción de indicadores reporta cada país?",
    xaxis_title="% de indicadores con al menos un dato",
    yaxis_title="País (código ISO3)",
    height=3200,
    margin=dict(b=110),
)

fig_cobertura_pais_onu.add_annotation(
    text=(
        "Nota: cada barra representa uno de los 193 Estados miembros de la ONU. El % indica<br>"
        "cuántos de los 974 indicadores candidatos tienen al menos un dato para ese país (1960-2025)."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.03,
    showarrow=False,
    font=dict(size=13, color="gray"),
    align="left",
)

figuras_reporte.append(("cobertura_por_pais_onu", fig_cobertura_pais_onu))
fig_cobertura_pais_onu.show()

## Curva de cobertura agregada por año con universo ONU

In [33]:
celdas_posibles_anio_onu = n_paises_total_onu * n_indicadores_total

densidad_anio_onu = (
    wdi_long_onu.dropna(subset=["Value"])
    .groupby("Year")
    .size()
    .reindex(anios_totales, fill_value=0)
    / celdas_posibles_anio_onu
)

df_curva_anio_onu = densidad_anio_onu.reset_index()
df_curva_anio_onu.columns = ["Year", "densidad"]

fig_curva_anio_onu = px.line(
    df_curva_anio_onu,
    x="Year",
    y="densidad",
    labels={"densidad": "% de celdas país-indicador con dato"},
)

fig_curva_anio_onu.update_layout(
    title="Curva agregada de cobertura por año (universo ONU, 193 países)",
    margin=dict(b=110),
)

fig_curva_anio_onu.add_annotation(
    text=(
        "Nota: % de celdas posibles (193 países × 974 indicadores) con dato, para cada año.<br>"
        "Comparar con la versión de 217 países para ver si el 'codo' de la curva se desplaza."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.25,
    showarrow=False,
    font=dict(size=11, color="gray"),
    align="left",
)

figuras_reporte.append(("curva_cobertura_anio_onu", fig_curva_anio_onu))
fig_curva_anio_onu.show()

Una diferencia que encontramos cuando hacemos este análisis solo con miembros ONU, es que en los años 90 la cobertura sube a 0.37, mientras que cuando considerábamos las unidades económicas en general (las 217), la cobertura llegaba a 0.35. Aunque no es mucha diferencia, se puede apreciar una mejora en la captación de los datos. Esto tiene sentido dado que la porción excluida representa a penas un 11% del total considerado al incio.

## Dispersión de cobertura por región y subregión (M49)

In [34]:
fig_dispersion_region_onu = px.box(
    df_pais_plot_onu,
    x="Region Name",
    y="pct_indicadores_con_dato",
    points="all",
    hover_data=["Country Code", "n_indicadores_con_dato", "Sub-region Name"],
    labels={"pct_indicadores_con_dato": "% de indicadores con al menos un dato"},
)

fig_dispersion_region_onu.update_layout(
    title="Dispersión de cobertura por región M49 (Region Name): ¿hay regiones sistemáticamente peor cubiertas?",
    xaxis_title="Región (M49 — continental)",
    yaxis_title="% de indicadores con al menos un dato",
    margin=dict(b=150),
)

fig_dispersion_region_onu.add_annotation(
    text=(
        "Nota: cada punto es un Estado miembro de la ONU (193); la caja muestra mediana y rango<br>"
        "intercuartílico de cobertura de indicadores dentro de cada región continental M49."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.35,
    showarrow=False,
    font=dict(size=10, color="gray"),
    align="left",
)

figuras_reporte.append(("dispersion_cobertura_region_m49", fig_dispersion_region_onu))
fig_dispersion_region_onu.show()

En línea con lo anterior, aquí podemos apreciar que la caja con datos extraídos de oceanía para la base de datos del WDI (Q1 a Q3) se ubica por debajo de los restantes continentes. Sin embargo, el 50% de sus países tienen una cobertura entre el 66 y 88 %. Los demás continentes están más concentrados, y todos tienen sus rangos intercuartílicos sobre 79%.


In [39]:
fig_dispersion_subregion_onu = px.box(
    df_pais_plot_onu,
    x="Sub-region Name",
    y="pct_indicadores_con_dato",
    points="all",
    hover_data=["Country Code", "n_indicadores_con_dato", "Region Name"],
    labels={"pct_indicadores_con_dato": "% de indicadores con al menos un dato"},
)

fig_dispersion_subregion_onu.update_layout(
    title="Dispersión de cobertura por subregión M49 (Sub-region Name)",
    xaxis_title="Subregión (M49)",
    yaxis_title="% de indicadores con al menos un dato",
    margin=dict(b=220),
)
fig_dispersion_subregion_onu.update_xaxes(tickangle=45)

fig_dispersion_subregion_onu.add_annotation(
    text=(
        "Nota: cada punto es un Estado miembro de la ONU (193); algunas subregiones tienen muy<br>"
        "pocos países (ej. Micronesia, Polynesia) — leer su dispersión con cautela por bajo n."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.70,
    showarrow=False,
    font=dict(size=13, color="gray"),
    align="left",
)

figuras_reporte.append(("dispersion_cobertura_subregion_m49", fig_dispersion_subregion_onu))
fig_dispersion_subregion_onu.show()

## Heatmap con cobertura regional en el tiempo

In [40]:
wdi_long_region_onu = wdi_long_onu.merge(
    df_regiones_onu[["ISO-alpha3", "Region Name"]],
    left_on="Country Code", right_on="ISO-alpha3", how="left"
)

n_paises_por_region_onu = df_regiones_onu.groupby("Region Name")["ISO-alpha3"].nunique()
celdas_posibles_region_onu = n_paises_por_region_onu * n_indicadores_total

celdas_con_dato_region_anio_onu = (
    wdi_long_region_onu.dropna(subset=["Value"])
    .groupby(["Year", "Region Name"])
    .size()
    .unstack("Region Name")
)

densidad_region_anio_onu = celdas_con_dato_region_anio_onu.div(celdas_posibles_region_onu, axis=1).fillna(0)
matriz_region_anio_onu = densidad_region_anio_onu.T

fig_heatmap_region_onu = px.imshow(
    matriz_region_anio_onu,
    labels=dict(x="Año", y="Región (M49)", color="% celdas con dato"),
    x=matriz_region_anio_onu.columns,
    y=matriz_region_anio_onu.index,
    color_continuous_scale="RdYlGn",
    zmin=0,
    zmax=float(matriz_region_anio_onu.values.max()),
    aspect="auto",
)

fig_heatmap_region_onu.update_layout(
    title="Densidad regional M49 en el tiempo (Region Name): ¿dónde y cuándo se concentran los huecos?",
    margin=dict(b=130),
)

fig_heatmap_region_onu.add_annotation(
    text=(
        "Nota: cada celda muestra, para esa región M49 y ese año, qué % de las celdas país-<br>"
        "indicador posibles (países ONU de la región × 974 indicadores) tienen realmente un dato."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.45,
    showarrow=False,
    font=dict(size=13, color="gray"),
    align="left",
)

figuras_reporte.append(("densidad_regional_heatmap_m49", fig_heatmap_region_onu))
fig_heatmap_region_onu.show()

Este gráfico nos permite ver que desde los años 90 en adelante obtenemos mejor cobertura. Sin embargo, también nos permite ver que para los territorios de Oceanía, esta cobertura inicio con mayor retraso en el tiempo. Para el año 1995, por ejemplo, mientras que para este continente la cobertura es del 33%, para los restantes es superior al 45%. Esto quiere decir que la distribución de datos faltantes no es aleatoria, sino que está concentrada territorialmente.

## Distribución de porcentajes de cobertura para miembros ONU

In [42]:
df_plot_onu = disponibilidad_indicador_full_onu.reset_index().rename(columns={"index": "Código WDI"})

fig_cobertura_indicador_onu = px.histogram(
    df_plot_onu,
    x="pct_paises_con_dato_onu",
    nbins=50,
    hover_data=["Código WDI", "n_paises_con_dato_onu"],
    labels={"pct_paises_con_dato_onu": "% de países ONU con al menos un dato"},
)

fig_cobertura_indicador_onu.update_layout(
    title="Cobertura por indicador (universo ONU, 193 países): ¿a cuántos países alcanza cada variable?",
    xaxis_title="% de países ONU con al menos un dato",
    yaxis_title="Cantidad de indicadores",
    bargap=0.05,
    margin=dict(b=110),
)

fig_cobertura_indicador_onu.add_annotation(
    text=(
        "Nota: cada barra agrupa indicadores según el % de los 193 Estados miembros de la ONU<br>"
        "que reportan al menos un dato en algún año (1960-2025). n=974 indicadores candidatos."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.30,
    showarrow=False,
    font=dict(size=13, color="gray"),
    align="left",
)

figuras_reporte.append(("cobertura_por_indicador_onu", fig_cobertura_indicador_onu))
fig_cobertura_indicador_onu.show()

# Finalmente vamos calcular cuanto se redujo la dispersión con el recorte del universo a paises miembros de ONU

In [43]:
comparacion_dispersión = pd.DataFrame({
    "Universo WDI (217 países)": disponibilidad_indicador_full["pct_paises_con_dato"].describe(),
    "Universo ONU (193 países)": disponibilidad_indicador_full_onu["pct_paises_con_dato_onu"].describe(),
})

comparacion_dispersión.loc["IQR"] = comparacion_dispersión.loc["75%"] - comparacion_dispersión.loc["25%"]
comparacion_dispersión

,Universo WDI (217 países),Universo ONU (193 países)
count,974.000000,974.000000
mean,0.807772,0.857550
std,0.180851,0.176621
min,0.004608,0.000000
25%,0.737327,0.818653
50%,0.861751,0.922280
75%,0.935484,0.979275
max,1.000000,1.000000
IQR,0.198157,0.160622


### Interpretación
pct_paises_con_dato es una columna de disponibilidad_indicador_full, con 974 filas, una por indicador. Cada valor de esa columna ya es en sí mismo un porcentaje (para ese indicador puntual, qué % de países lo reportan). Entonces .describe() no calcula "el % de cobertura general" — calcula estadísticas descriptivas sobre esos 974 porcentajes, es decir, resume cómo se distribuye la cobertura entre indicadores.

### Algunas referencias:
count:	Cuántos indicadores entran en el cálculo (974 en ambas columnas, ya que la reducción la aplicamos en los países, no en los indicadores)

mean:	El promedio de cobertura entre los 974 indicadores — ej. si da 0.80, significa que "en promedio, un indicador cubre 75% de los países del universo"
En este sentido, hemos hecho una mejora del 5%. 

IQR: 75% - 25%, el rango donde cae el 50% "central" de los indicadores. Lo agregamos porque mide dispersión de forma más robusta que std porque no se deja arrastrar por valores extremos

In [45]:
df_comparacion = pd.concat([
    pd.DataFrame({
        "pct_cobertura": disponibilidad_indicador_full["pct_paises_con_dato"],
        "Universo": "WDI (217 países)",
    }),
    pd.DataFrame({
        "pct_cobertura": disponibilidad_indicador_full_onu["pct_paises_con_dato_onu"],
        "Universo": "ONU (193 países)",
    }),
])

fig_comparacion_universos = px.box(
    df_comparacion,
    x="Universo",
    y="pct_cobertura",
    points="outliers",
    labels={"pct_cobertura": "% de países con al menos un dato"},
)

fig_comparacion_universos.update_layout(
    title="Comparación de cobertura por indicador: universo WDI vs. universo ONU",
    xaxis_title="Universo de países considerado",
    yaxis_title="% de países con al menos un dato",
    margin=dict(b=110),
)

fig_comparacion_universos.add_annotation(
    text=(
        "Nota: cada caja resume la distribución de cobertura de los mismos 974 indicadores,<br>"
        "calculada sobre dos universos de países distintos (217 unidades WDI vs. 193 Estados ONU)."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.25,
    showarrow=False,
    font=dict(size=13, color="gray"),
    align="left",
)

figuras_reporte.append(("comparacion_universos_wdi_onu", fig_comparacion_universos))
fig_comparacion_universos.show()